# 02 Deconvolution Quality

Inspect component weights, entropy, and class-component structure.

In [ ]:
from pathlib import Path
import os
import sys
import json

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root containing src/cytof_archetypes')

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
def _resolve_out_dir() -> Path:
    env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
    if env:
        return Path(env)
    cfg_path = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
    if cfg_path.exists():
        try:
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            out_raw = cfg.get('output_dir')
            if out_raw:
                out_path = Path(out_raw)
                return out_path if out_path.is_absolute() else REPO_ROOT / out_path
        except Exception:
            pass
    return REPO_ROOT / 'outputs' / 'experiment_suite'

OUT_DIR = _resolve_out_dir()

def _artifact_exists(path: Path) -> bool:
    if path.exists():
        return True
    print(f'Missing artifact: {path}')
    print('Run the suite first: python scripts/run_experiment_suite.py --config configs/experiment_suite.yaml')
    return False
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repo root:', REPO_ROOT)
print('Using src dir:', SRC_DIR)
print('Using suite output dir:', OUT_DIR)


In [ ]:
import pandas as pd
deconv_path = OUT_DIR / 'tables' / 'deconvolution_quality_summary.csv'
class_path = OUT_DIR / 'tables' / 'class_component_means.csv'
entropy_path = OUT_DIR / 'tables' / 'per_cell_weight_entropy.csv'
if _artifact_exists(deconv_path):
    deconv = pd.read_csv(deconv_path)
    display(deconv.sort_values(['method','k','seed']).head(20))

In [ ]:
if _artifact_exists(class_path):
    class_means = pd.read_csv(class_path)
    display(class_means.head())
if _artifact_exists(entropy_path):
    entropy = pd.read_csv(entropy_path)
    display(entropy.groupby('method')['weight_entropy'].describe())